# Task 3 — ANOVA Analysis
## Question 14: Testing Differences in Median House Value (MEDV) Across Age Groups

### Background

We use **ANOVA (Analysis of Variance)** to test whether there is a statistically significant difference in the **median value of houses (MEDV)** among the following three age groups:

- **Group 1:** 35 years and younger
- **Group 2:** Between 35 and 70 years
- **Group 3:** 70 years and older

---

### Hypotheses of ANOVA

$$H_0: \mu_1 = \mu_2 = \mu_3 \quad \text{(The means of MEDV are equal across all three age groups)}$$

$$H_a: \text{At least one age group has a different mean MEDV}$$

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("Libraries loaded successfully.")

In [ ]:
# Load the Boston Housing dataset
# You can load it from sklearn or from a CSV file
try:
    from sklearn.datasets import load_boston
    boston = load_boston()
    df = pd.DataFrame(boston.data, columns=boston.feature_names)
    df['MEDV'] = boston.target
except Exception:
    # Alternative: load from a URL
    url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
    df = pd.read_csv(url)
    df.columns = [c.upper() for c in df.columns]

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# The AGE column: proportion of owner-occupied units built prior to 1940
# We split it into three groups

def age_group(age):
    if age <= 35:
        return 'Group 1: ≤35'
    elif age <= 70:
        return 'Group 2: 35–70'
    else:
        return 'Group 3: >70'

df['AGE_GROUP'] = df['AGE'].apply(age_group)

print("Group counts:")
print(df['AGE_GROUP'].value_counts())

In [ ]:
# Descriptive statistics by group
desc = df.groupby('AGE_GROUP')['MEDV'].agg(['count', 'mean', 'median', 'std'])
desc.columns = ['Count', 'Mean', 'Median', 'Std Dev']
print("Descriptive Statistics of MEDV by Age Group:")
desc.round(3)

In [ ]:
# Visualization: Boxplot
plt.figure(figsize=(8, 5))
order = ['Group 1: ≤35', 'Group 2: 35–70', 'Group 3: >70']
sns.boxplot(data=df, x='AGE_GROUP', y='MEDV', order=order, palette='pastel')
plt.title('Distribution of MEDV by Age Group', fontsize=14)
plt.xlabel('Age Group')
plt.ylabel('Median House Value (MEDV, $1000s)')
plt.tight_layout()
plt.show()

### Step 1 — State the Hypotheses

| | |
|---|---|
| **H₀** | The means of MEDV are **equal** across all three age groups: $\mu_1 = \mu_2 = \mu_3$ |
| **Hₐ** | **At least one** age group has a different mean MEDV |

In [ ]:
# Step 2 — Check Assumption: Normality (Shapiro-Wilk test)
print("Shapiro-Wilk Normality Test for MEDV within each group:\n")
for grp in order:
    subset = df.loc[df['AGE_GROUP'] == grp, 'MEDV']
    stat, p = stats.shapiro(subset)
    print(f"{grp}: W = {stat:.4f}, p-value = {p:.4f}")

In [ ]:
# Step 3 — Check Assumption: Equality of Variances (Levene's test)
g1 = df.loc[df['AGE_GROUP'] == 'Group 1: ≤35',   'MEDV']
g2 = df.loc[df['AGE_GROUP'] == 'Group 2: 35–70', 'MEDV']
g3 = df.loc[df['AGE_GROUP'] == 'Group 3: >70',   'MEDV']

lev_stat, lev_p = stats.levene(g1, g2, g3)
print(f"Levene's Test: statistic = {lev_stat:.4f}, p-value = {lev_p:.4f}")
if lev_p > 0.05:
    print("→ Variances are equal (fail to reject H₀ of Levene's test).")
else:
    print("→ Variances are NOT equal (reject H₀ of Levene's test).")

In [ ]:
# Step 4 — One-Way ANOVA
f_stat, p_value = stats.f_oneway(g1, g2, g3)

alpha = 0.05
print("=" * 45)
print("         One-Way ANOVA Results")
print("=" * 45)
print(f"  F-statistic : {f_stat:.4f}")
print(f"  p-value     : {p_value:.6f}")
print(f"  Alpha       : {alpha}")
print("-" * 45)
if p_value < alpha:
    print("  Decision    : REJECT H₀")
    print("  Conclusion  : At least one age group has a")
    print("                significantly different mean MEDV.")
else:
    print("  Decision    : FAIL TO REJECT H₀")
    print("  Conclusion  : No significant difference in mean")
    print("                MEDV across age groups.")
print("=" * 45)

In [ ]:
# Step 5 — Post-hoc Analysis: Tukey HSD (if H₀ was rejected)
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df['MEDV'], groups=df['AGE_GROUP'], alpha=0.05)
print(tukey.summary())

### Summary

| Step | Description |
|------|-------------|
| **H₀** | Means of MEDV are equal across all three age groups |
| **Hₐ** | At least one age group has a different mean MEDV |
| **Test** | One-Way ANOVA (`scipy.stats.f_oneway`) |
| **Post-hoc** | Tukey HSD (if ANOVA is significant) |

> **Correct answer for Question 16:**  
> **H₀: The means of MEDV are equal across all three age groups**  
> **Hₐ: At least one age group has a different mean MEDV**
>
> ANOVA tests differences in **means**, not medians, variances, or distributions.